In [ ]:
import sys
sys.path.append('../')

import torch
import random
import torch.backends.cudnn as cudnn
import numpy as np
from utils.dataset_cfd import GraphDataset_paired, GraphDataset_unpaired
from train.train_MGN_cfd import train_MGN_comp, train_MGN_sup
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
np.random.seed(0)
cudnn.benchmark = False
cudnn.deterministic = True
random.seed(0)

In [ ]:
device="cuda:0"

In [ ]:
data_dir="../data/"
result_dir="../results/MGN_cfd/"

In [ ]:
idx_train=np.random.choice(300, 200, replace=False)
idx_test=np.array([i for i in range(300) if i not in idx_train])


train1=GraphDataset_paired(idx_train[:40], data_dir, "cpu")
train2=GraphDataset_unpaired(idx_train[40:], data_dir, "cpu")
train=GraphDataset_paired(idx_train, data_dir, "cpu")
test1=GraphDataset_paired(idx_test, data_dir, "cpu")

In [ ]:
train_MGN_sup(device, train, test1, result_dir, ib_n=False, ib_e=False, num_exp=1)
#sup: fully supervised baseline
#ib: inductive bias
#ib_n: node-level centering 
#ib_e: message-level centering
#num_exp: number of experiments with the same seed

In [ ]:
train_MGN_comp(device, train1, train2, test1, result_dir, ib_n=True, ib_e=True, num_exp=1)
#comp: complementary learning

In [ ]:
from utils.analysis import RMSE

print(RMSE(result_dir=result_dir, model="MGN", learning="sup", N_paired=200, N_total=200, ib="FF", exp_list=[0]))
print(RMSE(result_dir=result_dir, model="MGN", learning="comp", N_paired=20, N_total=200, ib="TT", exp_list=[0]))

#print (mean, std)

In [ ]:
from models.MGN_cfd import MGN_shared, Decoder_F

depth=3
y_input_size=1
pos_input_size=2
edge_input_size=4
hidden_size=150
output_size=1
residual=True
ib_n=True 
ib_e=True 
model_shared=MGN_shared(depth, y_input_size, pos_input_size, edge_input_size, hidden_size, ib_n, ib_e)
model_shared.load_state_dict(torch.load("../results/cfd/shared_MGN_comp_20_200_TT_0.pt"))
model_F= Decoder_F(hidden_size, output_size, residual)
model_F.load_state_dict(torch.load("../results/cfd/F_MGN_comp_20_200_TT_0.pt"))

model_shared_sup=MGN_shared(depth, y_input_size, pos_input_size, edge_input_size, hidden_size, ib_n=False, ib_e=False)
model_shared_sup.load_state_dict(torch.load("../results/cfd/shared_MGN_sup_200_200_FF_0.pt"))
model_F_sup= Decoder_F(hidden_size, output_size, residual)
model_F_sup.load_state_dict(torch.load("../results/cfd/F_MGN_sup_200_200_FF_0.pt"))

In [ ]:
import matplotlib.pyplot as plt


train = GraphDataset_paired(np.array(list(range(300))), data_dir,"cpu")

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
plt.subplots_adjust(wspace=0.01, hspace=0.01)  

for row, i in enumerate(np.sort(idx_test)[np.array(list(range(60,100,10)))]):
    l_pos, l_y, l_e, h_pos, h_e, y = train[i]

    # Low-res input
    axes[row, 0].imshow(l_y.numpy().reshape(32, 32), origin="lower")
    axes[row, 0].set_title("")
    axes[row, 0].axis("off")

    # Ground truth
    axes[row, 1].imshow(y.numpy().reshape(1024, 1024), origin="lower")
    axes[row, 1].set_title("")
    axes[row, 1].axis("off")

    # Prediction
    emb = model_shared(l_pos, l_y, l_e, h_pos, h_e)
    out = model_F(emb, l_y, l_pos, h_pos)
    axes[row, 2].imshow(out.detach().cpu().numpy().reshape(1024, 1024), origin="lower")
    axes[row, 2].set_title("")
    axes[row, 2].axis("off")

    # Prediction
    emb = model_shared_sup(l_pos, l_y, l_e, h_pos, h_e)
    out = model_F_sup(emb, l_y, l_pos, h_pos)
    axes[row, 3].imshow(out.detach().cpu().numpy().reshape(1024, 1024), origin="lower")
    axes[row, 3].set_title("")
    axes[row, 3].axis("off")


plt.show()